# Master Science et Technologie de la santé - Parcours Santé Mentale Numérique

**(UE5) Stratégies thérapeutiques innovantes pour la santé mentale**

Content creator: Zaineb Amor, GHU Paris Psychiatrie et Neurosciences, INM


# MRI and fMRI Data Processing for Neuronavigated and fMRI-based TMS

This notebook outlines the workflow for processing **MRI** and **fMRI** data for **neuronavigated** and **fMRI-based TMS**.

## Workflow Steps

- **Preprocessing** is performed using **fMRIPREP**, **CHARM-TMS**, and custom Python code.  
  *Note: The fMRIPREP and CHARM-TMS steps have already been completed to save time.*
  
1. **fMRIPREP** – preprocessing anatomical and functional images  
2. **Denoising** – reducing noise from the fMRI data  
3. **h5py → txt conversion** – preparing data for further analysis  
4. **CHARM-TMS** – build head models 

- **Target Selection**: The choice of stimulation target depends on the condition:  **SCZ** – targets are selected using BrainVoyager  

- **Target creation and export**: Finally, the selected target is **created and exported** in a format compatible with TMS machines.



In [1]:
import os
import glob
import sys

current_path = os.getcwd()          # Get current working directory
print("Current:", current_path)
parent_path = os.path.dirname(current_path)   # Go one folder up
print("Parrent:", parent_path)
os.chdir('../..')
sys.path.append(parent_path)

Current: /home/zamor/Documents/rTMS_DomenechAmor_2025/Codes/rsTMS_pipeline/notebooks
Parrent: /home/zamor/Documents/rTMS_DomenechAmor_2025/Codes/rsTMS_pipeline


In [2]:
from rsTMS_pipeline.data_loading.params import *
from rsTMS_pipeline.data_loading.loading_utils import *
from rsTMS_pipeline.preproc.preproc_utils import *
from rsTMS_pipeline.targeting.targeting_utils import *
from rsTMS_pipeline.plotting.plotting_utils import *

from nilearn import image, plotting
from nilearn.interfaces.fmriprep import load_confounds
from nilearn.image import mean_img, load_img, clean_img,index_img
import numpy as np
#from simnibs import opt_struct, mni2subject_coords
#from simnibs import localite


**Denoising Functional MRI Data**
  
  For each subject and session:
   - Load the preprocessed anatomical (T1w) and functional (BOLD) images, as well as the brain mask.
   - Compute the mean functional image for inspection.
   - Load confound regressors (motion parameters, white matter and CSF signals) to remove noise.
   - Clean the functional data using the confounds and brain mask, applying linear detrending but without standardization.
   - Compute the mean of the cleaned functional image for quality check.
   - Save the cleaned functional image with a new filename (replacing "preproc_bold" with "preproc_bold_cleaned").
 
 Note:
   - The cleaning strategy is based on motion and WM/CSF confounds, which helps reduce physiological and motion-related artifacts.


In [3]:
for subject in subjects:
    for session in sessions:
        
        ANAT_FILE = f'sub-{subject:02}_ses-{session}_space-MNI152NLin2009cAsym_desc-preproc_T1w.nii.gz'
        FUNC_FILE = f'sub-{subject:02}_ses-{session}_task-rs_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz' 
        MASK_FILE = f'sub-{subject:02}_ses-{session}_task-rs_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz'
        ANAT_PATH = os.path.join(FMRIPREP_PATH, f'sub-{subject:02}',f'ses-{session}', 'anat', ANAT_FILE)
        FUNC_PATH = os.path.join(FMRIPREP_PATH, f'sub-{subject:02}',f'ses-{session}', 'func', FUNC_FILE)
        MASK_PATH = os.path.join(FMRIPREP_PATH, f'sub-{subject:02}',f'ses-{session}', 'func', MASK_FILE)

        anat = image.load_img(ANAT_PATH)
        func = image.load_img(FUNC_PATH)
        mean_img=image.mean_img(func)
        mask = image.load_img(MASK_PATH)

        confounds,sample_mask=load_confounds(FUNC_PATH,strategy=("motion","wm_csf"),motion="derivatives")
        func_cleaned=image.clean_img(func,confounds=confounds,sample_mask=sample_mask,mask_img=mask,standardize=False,linear_detrend=True)
        cleaned_mean_img=image.mean_img(func_cleaned)
        CFUNC_PATH=FUNC_PATH.replace("preproc_bold","preproc_bold_cleaned")
        func_cleaned.to_filename(CFUNC_PATH)


/home/team/.pyenv/versions/3.11.6/envs/compbrain/lib/python3.11/site-packages/nilearn/image/image.py:1272: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.13, the default strategy will be replaced by the new strategy and the 'zscore' option will be removed. Please use 'zscore_sample' instead.
  data = signal.clean(


**Extract and Save Affine Transforms from fMRIPREP H5 Files**
    
For each subject and session:
   - Locate the H5 file that stores the transformation from MNI space to the subject's T1w anatomical space.
   - Open the H5 file using h5py and extract:
       * TransformParameters: contains the rotation matrix and translation vector.
       * TransformFixedParameters: contains the center of rotation.
   - Reshape the first 9 parameters into a 3x3 rotation matrix.
   - Extract the translation vector (parameters 10-12) and the center.
   - Initialize a MatrixOffsetTransformBase object with the matrix, translation, and center.
   - Compute the offset and generate the full affine transformation matrix.
   - Print all intermediate results for verification:
       * Rotation matrix
       * Translation vector
       * Center
       * Offset
       * Affine matrix
   - Check if the output folder exists in TRANSFORM_PATH; if not, create it.
   - Save the affine matrix as a .txt file (converted from .h5) for downstream use.

 Note:
   - This step allows the affine transform from template (MNI) to individual anatomical space to be exported in a standard format.

In [4]:
for subj in subjects:
    for ses in sessions: 
        transform_file = glob.glob(os.path.join(FMRIPREP_PATH, f'sub-{subj:02}',f'ses-{ses}', 'anat', "*from-MNI152NLin2009cAsym_to-T1w_mode-image_xfm.h5"), recursive=True)[0]
        print(transform_file)
        f = h5py.File(transform_file, 'r')
        transform_parameters = f['TransformGroup']['2']['TransformParameters']
        fixed_parameters = f['TransformGroup']['2']['TransformFixedParameters']
        matrix=np.reshape(transform_parameters[:9], (3, 3))
        print(f'Matrix = \n{matrix}\n')
        translation=transform_parameters[9:12]
        print(f'Translation vector = \n{translation}\n')
        center=fixed_parameters[()]
        print(f'Center = \n{center}\n')
        transform = MatrixOffsetTransformBase(matrix, translation, center)
        transform.compute_offset()
        print(f'Offset = \n{transform.offset}\n')
        transform.generate_affine_matrix()
        print(f'Affine matrix = \n{transform.affine_matrix}\n')
        if not os.path.exists(os.path.join(TRANSFORM_PATH, f'sub-{subj:02}',f'ses-{ses}')):
            print(f"Folder '{os.path.join(TRANSFORM_PATH, f'sub-{subj:02}',f'ses-{ses}')}' does not exist. Creating it...")
            os.makedirs(os.path.join(TRANSFORM_PATH, f'sub-{subj:02}',f'ses-{ses}'))
        else:
            print(f"Folder '{os.path.join(TRANSFORM_PATH, f'sub-{subj:02}',f'ses-{ses}')}' already exists.") 
        np.savetxt(os.path.join(os.path.join(TRANSFORM_PATH, f'sub-{subj:02}',f'ses-{ses}'),
                                os.path.basename(transform_file).replace('_ses-pre','').replace('h5','txt')),transform.affine_matrix, delimiter=' ')

/home/zamor/Documents/MainStim/derivatives/fmriprep/sub-08/ses-1/anat/sub-08_ses-1_from-MNI152NLin2009cAsym_to-T1w_mode-image_xfm.h5
Matrix = 
[[ 1.0766052e+00 -7.4966937e-02 -1.1260718e-02]
 [-1.9309595e-02  9.8946363e-01 -3.1906095e-01]
 [ 9.9408627e-04  4.0108085e-01  1.0623643e+00]]

Translation vector = 
[-3.7268853 13.577996  37.09256  ]

Center = 
[ 0.23592781 21.99948502  9.74520111]

Offset = 
[-1.98598664 16.92365984 27.66100017]

Affine matrix = 
[[ 1.07660520e+00  7.49669373e-02 -1.12607181e-02 -1.98598664e+00]
 [ 1.93095952e-02  9.89463627e-01 -3.19060951e-01  1.69236598e+01]
 [ 9.94086266e-04  4.01080847e-01  1.06236434e+00 -2.76610002e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Folder '/home/zamor/Documents/MainStim/derivatives/h5_transforms/sub-08/ses-1' already exists.


**TMS Optimization Using CHARM-TMS and SimNIBS**
- **Initialize TMS optimization** object: `TMSoptimize()`
- Set **subject folder** in CHARM-TMS derivatives
- Set **output folder** for SimNIBS FEM results
- Select **coil model** (`.ccd` file) required for the ADM method
- Define the **target** for stimulation using MNI coordinates converted to subject space (`mni2subject_coords`)
- Choose the **ADM method** for coil placement optimization
- **Run the optimization** to compute the best coil position
- Save the optimized coil position to a file for later neuronavigation

Notes

- Target coordinates: `mni_coords = (-64, -24, 36)` (example)
- Output files are stored in subject/session-specific folders
- This workflow ensures precise TMS targeting in subject-specific space

In [ ]:
mni_coords = (-60,-24,36)
# Initialize structure
tms_opt = opt_struct.TMSoptimize()

# Subject folder
for subject in subjects:
    for session in sessions: 
        tms_opt.subpath = os.path.join(CHARM_PATH, f'm2m_sub-{subject:02}_ses-{session}')
        print(tms_opt.subpath)
        # Select output folder
        tms_opt.pathfem = os.path.join(SIMNIBS_PATH, f'sub-{subject:02}_ses-{session}_tmsoptim')
        print(tms_opt.pathfem)
        # Select the coil model
        # The ADM method requires a '.ccd' coil model
        tms_opt.fnamecoil ='/home/zaineb/simnibs/resources/coil_models/Drakaki_BrainStim_2022/MagVenture_Cool-B65.ccd'
        # Select a target for the optimization
        tms_opt.target = mni2subject_coords(mni_coords, tms_opt.subpath)
        print('Target:', tms_opt.target, 'End Target')
        # Use the ADM method
        tms_opt.method = 'ADM'
        # Run optimization
        opt_pos=tms_opt.run()
        fn = os.path.join(tms_opt.pathfem, f'sub-{subject:02}_ses-{session}_opt_pos')
        localite().write(np.squeeze(opt_pos), fn)